# Exploration de `_determine_imputation_order`

Ce notebook explore le comportement de la méthode `_determine_imputation_order` du `HighFrequencyImputer` dans trois scénarios :

1. **Séries temporelles simples** — fréquences mixtes sans dimension de panel
2. **Panel homogène** — même fréquence par indicateur pour toutes les entités
3. **Panel hétérogène** — fréquences différentes par entité, avec asymétrie basse/haute fréquence

Seules les fréquences **mensuelle (M)**, **trimestrielle (Q)** et **annuelle (A)** sont utilisées.

---

## 0. Imports & utilitaires

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.linear_model import LinearRegression

from tsforecast.frequency.high_frequency_imputer import HighFrequencyImputer

In [ ]:
# Palette de couleurs par fréquence
FREQ_COLORS = {
    'A':  '#e15759',   # Annuelle  → rouge
    'QS': '#f28e2b',   # Trimestrielle → orange
    'Q':  '#f28e2b',
    'MS': '#4e79a7',   # Mensuelle → bleu
    'M':  '#4e79a7',
}

# Palette de couleurs par catégorie d'imputation
CAT_COLORS = {
    'impute':     '#59a14f',   # Vert  → à imputer
    'aggregate':  '#f28e2b',   # Orange → à agréger
    'target_freq':'#4e79a7',   # Bleu  → déjà à la bonne fréquence
}

FREQ_LABELS = {'A': 'Annuelle', 'QS': 'Trimestrielle', 'Q': 'Trimestrielle', 'MS': 'Mensuelle', 'M': 'Mensuelle'}
CAT_LABELS  = {'impute': 'À imputer', 'aggregate': 'À agréger', 'target_freq': 'Fréquence cible'}


def normalize_freq_label(freq: str) -> str:
    """Normalize frequency label for display."""
    mapping = {'MS': 'M', 'QS': 'Q', 'AS': 'A', 'YS': 'A', 'Y': 'A'}
    return mapping.get(freq, freq)


def print_section(title: str) -> None:
    """Print a formatted section title."""
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print(f"{'═' * 60}")

In [ ]:
def summarize_imputation_order(imputer: HighFrequencyImputer) -> pd.DataFrame:
    """Build a summary DataFrame from imputation_order_ and related attributes.

    Args:
        imputer: Fitted HighFrequencyImputer instance.

    Returns:
        DataFrame with columns: rank, key, variable, entity, frequency, category.
    """
    rows = []
    for rank, key in enumerate(imputer.imputation_order_, start=1):
        freq = imputer.detected_frequencies_.get(key, 'unknown')
        cat  = imputer.variable_categories_.get(key, 'unknown')

        if isinstance(key, tuple):
            entity   = key[:-1] if len(key) > 2 else key[0]
            variable = key[-1]
        else:
            entity   = None
            variable = key

        rows.append({
            'rang':     rank,
            'clé':      str(key),
            'variable': variable,
            'entité':   str(entity) if entity is not None else '—',
            'fréquence': normalize_freq_label(freq) if freq != 'unknown' else freq,
            'catégorie': cat,
        })
    return pd.DataFrame(rows)


def plot_imputation_order(imputer: HighFrequencyImputer, title: str, figsize=(14, 5)) -> None:
    """Visualize imputation order as a colored ranked bar chart.

    Args:
        imputer: Fitted HighFrequencyImputer instance.
        title: Plot title.
        figsize: Figure size.
    """
    df = summarize_imputation_order(imputer)
    if df.empty:
        print("Aucune variable à imputer.")
        return

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # --- Graphique 1: ordre d'imputation coloré par fréquence ---
    ax = axes[0]
    colors = [FREQ_COLORS.get(normalize_freq_label(imputer.detected_frequencies_.get(k, 'M')), '#aaa')
              for k in imputer.imputation_order_]
    labels = df['variable'] + (('\n(' + df['entité'] + ')') if (df['entité'] != '—').any() else '')
    bars = ax.barh(range(len(df)), df['rang'].max() - df['rang'] + 1,
                   color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Priorité (gauche = imputé en premier)', fontsize=9)
    ax.set_title('Ordre d\'imputation (par fréquence)', fontsize=11)
    ax.invert_yaxis()
    ax.set_xlim(0, df['rang'].max() + 0.5)
    # Numéros de rang
    for i, (bar, rang) in enumerate(zip(bars, df['rang'])):
        ax.text(bar.get_width() - 0.05, bar.get_y() + bar.get_height() / 2,
                f'#{rang}', va='center', ha='right', fontsize=9, color='white', fontweight='bold')
    # Légende fréquences
    freq_patches = [mpatches.Patch(color=FREQ_COLORS[f], label=FREQ_LABELS[f])
                    for f in ['A', 'Q', 'M'] if f in FREQ_COLORS]
    ax.legend(handles=freq_patches, fontsize=8, loc='lower right')

    # --- Graphique 2: catégories de toutes les variables ---
    ax2 = axes[1]
    cat_data = pd.Series(imputer.variable_categories_)
    counts = cat_data.value_counts()
    bar_colors = [CAT_COLORS.get(c, '#aaa') for c in counts.index]
    ax2.bar([CAT_LABELS.get(c, c) for c in counts.index], counts.values,
            color=bar_colors, edgecolor='white')
    ax2.set_title('Distribution des catégories de variables', fontsize=11)
    ax2.set_ylabel('Nombre de variables')
    for i, v in enumerate(counts.values):
        ax2.text(i, v + 0.1, str(v), ha='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()


def plot_freq_heatmap_panel(imputer: HighFrequencyImputer, title: str) -> None:
    """Plot a heatmap of variable frequencies per entity for panel data.

    Args:
        imputer: Fitted HighFrequencyImputer instance.
        title: Plot title.
    """
    # Construction de la table entité x variable
    records = {}
    for key, freq in imputer.detected_frequencies_.items():
        if isinstance(key, tuple):
            entity   = str(key[:-1][0] if len(key) == 2 else key[:-1])
            variable = key[-1]
        else:
            entity, variable = '—', key
        records[(entity, variable)] = normalize_freq_label(freq)

    df_heat = pd.Series(records).unstack(level=1)
    freq_order = ['A', 'Q', 'M']
    freq_num = {f: i for i, f in enumerate(freq_order)}
    num_map  = df_heat.applymap(lambda x: freq_num.get(x, np.nan) if pd.notna(x) else np.nan)

    fig, ax = plt.subplots(figsize=(max(8, len(df_heat.columns) * 1.2), max(4, len(df_heat) * 0.6)))
    cmap = plt.cm.get_cmap('RdYlBu', 3)
    im = ax.imshow(num_map.values, cmap=cmap, vmin=0, vmax=2, aspect='auto')

    ax.set_xticks(range(len(df_heat.columns)))
    ax.set_xticklabels(df_heat.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(df_heat.index)))
    ax.set_yticklabels(df_heat.index, fontsize=9)

    # Annotations dans chaque cellule
    for i in range(len(df_heat.index)):
        for j in range(len(df_heat.columns)):
            val = df_heat.values[i, j]
            if pd.notna(val):
                ax.text(j, i, val, ha='center', va='center', fontsize=9,
                        fontweight='bold', color='white')

    # Légende
    legend_patches = [
        mpatches.Patch(color=cmap(0), label='Annuelle (A)'),
        mpatches.Patch(color=cmap(0.5), label='Trimestrielle (Q)'),
        mpatches.Patch(color=cmap(1.0), label='Mensuelle (M)'),
    ]
    ax.legend(handles=legend_patches, loc='upper right', bbox_to_anchor=(1.25, 1), fontsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Variables', fontsize=10)
    ax.set_ylabel('Entités', fontsize=10)
    plt.tight_layout()
    plt.show()

---
## 1. Séries temporelles simples à fréquences mixtes

Le cas le plus simple : un seul "panel" implicite (pas d'entité), avec des variables à fréquences annuelle, trimestrielle et mensuelle. La fréquence cible est **mensuelle**.

**Comportement attendu** : `_determine_imputation_order` trie simplement les variables à imputer de la fréquence la plus basse vers la plus haute — annuelle en premier, trimestrielle ensuite.

In [ ]:
# Génération des dates sur 5 ans à fréquence mensuelle (index de référence)
dates_m = pd.date_range('2018-01-01', periods=60, freq='MS')

rng = np.random.default_rng(seed=42)

# Construction du DataFrame mensuel de base
# Les variables basses fréquences sont échantillonnées puis forward-fillées
df_ts = pd.DataFrame(index=dates_m)

# Variables mensuelles — directement disponibles à M
df_ts['var_m_1'] = rng.normal(100, 10, 60).cumsum() / 60 + 50
df_ts['var_m_2'] = rng.normal(0, 5, 60).cumsum() + 200
df_ts['var_m_3'] = rng.normal(0, 3, 60).cumsum() + 80

# Variables trimestrielles — valeur constante par trimestre, NaN entre les trimestres
dates_q = pd.date_range('2018-01-01', periods=20, freq='QS')
for name, scale in [('var_q_1', 200), ('var_q_2', 50), ('var_q_3', 300)]:
    s_q = pd.Series(rng.normal(scale, scale * 0.05, 20), index=dates_q)
    df_ts[name] = s_q.reindex(dates_m)  # NaN pour les mois non-début de trimestre

# Variables annuelles — valeur constante par année, NaN entre les années
dates_a = pd.date_range('2018-01-01', periods=5, freq='YS')
for name, scale in [('var_a_1', 1000), ('var_a_2', 500)]:
    s_a = pd.Series(rng.normal(scale, scale * 0.05, 5), index=dates_a)
    df_ts[name] = s_a.reindex(dates_m)  # NaN pour les mois non-janvier

print(f"Dimensions : {df_ts.shape}  |  Période : {dates_m[0].date()} → {dates_m[-1].date()}")
print(f"\nNombre de NaN par variable :")
print(df_ts.isna().sum().to_string())
df_ts.head(8)

In [ ]:
# Visualisation du jeu de données — disponibilité des observations
fig, ax = plt.subplots(figsize=(14, 4))

col_colors = (
    [FREQ_COLORS['M']] * 3 +
    [FREQ_COLORS['Q']] * 3 +
    [FREQ_COLORS['A']] * 2
)

for i, col in enumerate(df_ts.columns):
    mask = df_ts[col].notna()
    ax.scatter(df_ts.index[mask], [i] * mask.sum(),
               color=col_colors[i], s=20, alpha=0.8, linewidths=0)

ax.set_yticks(range(len(df_ts.columns)))
ax.set_yticklabels(df_ts.columns, fontsize=9)
ax.set_xlabel('Date')
ax.set_title('Disponibilité des observations — séries temporelles simples', fontweight='bold')
patches = [mpatches.Patch(color=FREQ_COLORS['A'], label='Annuelle'),
           mpatches.Patch(color=FREQ_COLORS['Q'], label='Trimestrielle'),
           mpatches.Patch(color=FREQ_COLORS['M'], label='Mensuelle')]
ax.legend(handles=patches, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Instanciation et fit de l'imputer
imputer_ts = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    cascade_refitting=False,
    imputation_scope='strict',
)

# Fit uniquement (pas besoin de transform pour inspecter l'ordre)
imputer_ts.fit(df_ts)

In [ ]:
print_section("Fréquences détectées")
for col, freq in imputer_ts.detected_frequencies_.items():
    cat = imputer_ts.variable_categories_.get(col, '?')
    print(f"  {col:<15} → {normalize_freq_label(freq):<5}  [{CAT_LABELS.get(cat, cat)}]")

print_section("Ordre d'imputation")
for rank, key in enumerate(imputer_ts.imputation_order_, start=1):
    freq = normalize_freq_label(imputer_ts.detected_frequencies_.get(key, '?'))
    print(f"  #{rank}  {key:<15}  (fréquence : {freq})")

In [ ]:
df_order_ts = summarize_imputation_order(imputer_ts)
display(df_order_ts)
plot_imputation_order(imputer_ts, title='Scénario 1 — Séries temporelles simples (cible : M)')

### 💡 Observations

- Les variables **annuelles** (`var_a_*`) apparaissent en **premier** : elles ont la fréquence la plus basse et sont donc imputées avant les autres.
- Les variables **trimestrielles** (`var_q_*`) viennent **ensuite**.
- Les variables **mensuelles** (`var_m_*`) sont catégorisées `target_freq` : elles n'entrent pas dans `imputation_order_`.
- La logique de tri est purement basée sur `get_frequency_order` — plus l'ordre est élevé (= fréquence basse), plus la variable est prioritaire.

---

## 2. Panel homogène — même fréquence par indicateur pour toutes les entités

Trois entités (`région_A`, `région_B`, `région_C`) avec exactement les **mêmes variables à les mêmes fréquences**. La fréquence cible est **mensuelle**.

**Comportement attendu** : les variables sont d'abord groupées par nom (même fréquence médiane), puis, à l'intérieur d'un groupe de variables, triées par fréquence locale. Comme toutes les entités ont les mêmes fréquences, l'ordre à l'intérieur du groupe est arbitraire (ou alphabétique).

In [ ]:
def build_entity_block(
    entity: str,
    dates_m: pd.DatetimeIndex,
    rng: np.random.Generator,
    var_specs: dict,
) -> pd.DataFrame:
    """Build a single-entity DataFrame with mixed-frequency variables.

    Args:
        entity: Entity identifier string.
        dates_m: Monthly DatetimeIndex (reference grid).
        rng: NumPy random Generator.
        var_specs: Dict mapping variable name to (freq_alias, base_value).

    Returns:
        DataFrame with MultiIndex (entity, date) and variable columns.
    """
    freq_map = {'M': 'MS', 'Q': 'QS', 'A': 'YS'}
    n_periods = {'M': len(dates_m), 'Q': len(dates_m) // 3 + 1, 'A': len(dates_m) // 12 + 1}

    df = pd.DataFrame(index=dates_m)
    df.index.name = 'date'

    for var, (freq, base) in var_specs.items():
        ref_dates = pd.date_range(dates_m[0], periods=n_periods[freq], freq=freq_map[freq])
        s = pd.Series(rng.normal(base, base * 0.05, len(ref_dates)), index=ref_dates)
        df[var] = s.reindex(dates_m)   # NaN aux mois sans observation

    df['entity'] = entity
    df = df.reset_index().set_index(['entity', 'date'])
    return df


dates_m = pd.date_range('2018-01-01', periods=60, freq='MS')
rng = np.random.default_rng(seed=0)

# Spécifications identiques pour toutes les entités
var_specs_homo = {
    'prod_indus':  ('M', 100),
    'emploi':      ('M', 500),
    'export':      ('M', 200),
    'pib':         ('Q', 1000),
    'invest':      ('Q', 300),
    'pop_active':  ('A', 5000),
}

entities_homo = ['région_A', 'région_B', 'région_C']
df_panel_homo = pd.concat([
    build_entity_block(e, dates_m, rng, var_specs_homo)
    for e in entities_homo
])

print(f"Dimensions : {df_panel_homo.shape}")
print(f"Entités    : {df_panel_homo.index.get_level_values('entity').unique().tolist()}")
print(f"Variables  : {df_panel_homo.columns.tolist()}")
df_panel_homo.head(10)

In [ ]:
# Visualisation — heatmap des données disponibles par entité
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, entity in zip(axes, entities_homo):
    sub = df_panel_homo.xs(entity, level='entity')
    im = ax.imshow(sub.T.notna().astype(int), aspect='auto',
                   cmap='Blues', vmin=0, vmax=1)
    ax.set_title(entity, fontweight='bold')
    ax.set_yticks(range(len(sub.columns)))
    ax.set_yticklabels(sub.columns, fontsize=8)
    ax.set_xlabel('Mois')
axes[0].set_ylabel('Variables')
fig.suptitle('Panel homogène — disponibilité des données (bleu = valeur disponible)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
imputer_homo = HighFrequencyImputer(
    target_frequency='M',
    estimator=LinearRegression(),
    cascade_refitting=False,
    imputation_scope='strict'
)

imputer_homo.fit(df_panel_homo)

In [ ]:
print_section("Fréquences détectées (panel homogène)")
for key, freq in sorted(imputer_homo.detected_frequencies_.items(), key=lambda x: str(x[0])):
    cat = imputer_homo.variable_categories_.get(key, '?')
    print(f"  {str(key):<35} → {normalize_freq_label(freq):<5}  [{CAT_LABELS.get(cat, cat)}]")

print_section("Ordre d'imputation (panel homogène)")
for rank, key in enumerate(imputer_homo.imputation_order_, start=1):
    freq = normalize_freq_label(imputer_homo.detected_frequencies_.get(key, '?'))
    print(f"  #{rank:02d}  {str(key):<35}  (fréquence : {freq})")

In [ ]:
df_order_homo = summarize_imputation_order(imputer_homo)
display(df_order_homo)

plot_freq_heatmap_panel(imputer_homo, 'Panel homogène — fréquences par entité × variable')
plot_imputation_order(imputer_homo, title='Scénario 2 — Panel homogène (cible : M)')

### 💡 Observations

- Toutes les entités ayant les **mêmes fréquences**, la fréquence médiane par variable est uniforme.
- `pop_active` (annuelle) est traitée **avant** `pib` / `invest` (trimestrielles) pour toutes les entités.
- À fréquence égale (ex. les deux variables Q), l'ordre interne est régi par le nombre d'entités affectées (critère de tri secondaire `n_entities` croissant), puis par la fréquence locale — ici identique pour toutes, donc l'ordre est stable mais peut varier selon l'implémentation.
- Dans ce scénario **homogène**, le résultat est similaire au cas de séries temporelles simples, simplement répété pour chaque entité.

---

## 3. Panel hétérogène — fréquences différentes par entité

Trois groupes d'entités avec des profils de données asymétriques :

| Entité | Profil |
|--------|--------|
| `pays_nord` | Beaucoup de variables **basse fréquence** (annuelles, trimestrielles) — peu de mensuelles |
| `pays_sud`  | Profil équilibré — mélange M/Q/A |
| `pays_est`  | Beaucoup de variables **haute fréquence** (mensuelles) — peu d'annuelles |

**Comportement attendu** : pour chaque variable partagée entre entités, `_determine_imputation_order` calcule la **médiane** des ordres de fréquence sur toutes les entités. Si `pib` est trimestrielle pour `pays_sud/est` mais annuelle pour `pays_nord`, sa fréquence médiane se situera entre les deux — ce qui peut réordonner les variables différemment du cas homogène.

In [ ]:
rng = np.random.default_rng(seed=7)

# Entité nord : beaucoup de variables basse fréquence
var_specs_nord = {
    'prod_indus': ('A', 100),   # mensuelle ailleurs, annuelle ici
    'emploi':     ('A', 500),   # mensuelle ailleurs, annuelle ici
    'export':     ('Q', 200),   # mensuelle ailleurs, trimestrielle ici
    'pib':        ('A', 1000),  # trimestrielle ailleurs, annuelle ici
    'invest':     ('Q', 300),   # trimestrielle
    'pop_active': ('A', 5000),  # annuelle partout
    'consomm':    ('Q', 800),   # nouvelle variable trimestrielle propre à nord
}

# Entité sud : profil équilibré
var_specs_sud = {
    'prod_indus': ('M', 110),
    'emploi':     ('M', 520),
    'export':     ('M', 210),
    'pib':        ('Q', 1050),
    'invest':     ('Q', 310),
    'pop_active': ('A', 4800),
    'tx_chomage': ('Q', 8),     # nouvelle variable propre à sud
}

# Entité est : beaucoup de variables haute fréquence
var_specs_est = {
    'prod_indus': ('M', 95),
    'emploi':     ('M', 480),
    'export':     ('M', 195),   # mensuelle
    'pib':        ('Q', 980),   # trimestrielle
    'invest':     ('M', 290),   # trimestrielle ailleurs, mensuelle ici
    'pop_active': ('A', 5100),  # annuelle partout
    'ipc':        ('M', 102),   # nouvelle variable mensuelle propre à est
}

df_panel_het = pd.concat([
    build_entity_block('pays_nord', dates_m, rng, var_specs_nord),
    build_entity_block('pays_sud',  dates_m, rng, var_specs_sud),
    build_entity_block('pays_est',  dates_m, rng, var_specs_est),
])

print(f"Dimensions : {df_panel_het.shape}")
print(f"Entités    : {df_panel_het.index.get_level_values('entity').unique().tolist()}")
print(f"Variables  : {df_panel_het.columns.tolist()}")
df_panel_het.head(10)

In [ ]:
# Résumé des fréquences prévues par entité
freq_preview = {
    'pays_nord': var_specs_nord,
    'pays_sud':  var_specs_sud,
    'pays_est':  var_specs_est,
}

all_vars = sorted(set(
    v for specs in freq_preview.values() for v in specs
))

df_preview = pd.DataFrame(
    {entity: {v: specs.get(v, ('—', 0))[0] for v in all_vars}
     for entity, specs in freq_preview.items()}
)
print("Fréquences par entité × variable (attendues) :")
display(df_preview)

In [ ]:
imputer_het = HighFrequencyImputer(
    target_frequency='Q',
    estimator=LinearRegression(),
    cascade_refitting=False,
    imputation_scope='strict'
)

imputer_het.fit(df_panel_het)

In [ ]:
print_section("Fréquences détectées (panel hétérogène)")
for key, freq in sorted(imputer_het.detected_frequencies_.items(), key=lambda x: str(x[0])):
    cat = imputer_het.variable_categories_.get(key, '?')
    print(f"  {str(key):<35} → {normalize_freq_label(freq):<5}  [{CAT_LABELS.get(cat, cat)}]")

print_section("Ordre d'imputation (panel hétérogène)")
for rank, key in enumerate(imputer_het.imputation_order_, start=1):
    freq = normalize_freq_label(imputer_het.detected_frequencies_.get(key, '?'))
    print(f"  #{rank:02d}  {str(key):<35}  (fréquence : {freq})")

In [ ]:
df_order_het = summarize_imputation_order(imputer_het)
display(df_order_het)

plot_freq_heatmap_panel(imputer_het, 'Panel hétérogène — fréquences par entité × variable')
plot_imputation_order(imputer_het, title='Scénario 3 — Panel hétérogène (cible : M)', figsize=(16, 7))

In [ ]:
# -----------------------------------------------------------------------
# Analyse détaillée : métriques de tri calculées par _determine_imputation_order
# -----------------------------------------------------------------------
from tsforecast.utils.frequency.utils import get_frequency_order, normalize_frequency

impute_keys = [k for k, cat in imputer_het.variable_categories_.items() if cat == 'impute']

# Reconstruction des métriques internes
var_to_freq_orders: dict = {}
for key in impute_keys:
    var_name = key[-1] if isinstance(key, tuple) else key
    freq = imputer_het.detected_frequencies_.get(key, 'M')
    var_to_freq_orders.setdefault(var_name, []).append(get_frequency_order(freq))

rows_metrics = []
for var, orders in var_to_freq_orders.items():
    rows_metrics.append({
        'variable':           var,
        'n_entités':          len(orders),
        'freq_médiane (ord)': round(np.median(orders), 2),
        'freq_min (ord)':     int(np.max(orders)),      # max ordre = min fréquence
        'freq_moyenne (ord)': round(np.mean(orders), 2),
    })

df_metrics = pd.DataFrame(rows_metrics).sort_values(
    by=['freq_médiane (ord)', 'freq_min (ord)', 'freq_moyenne (ord)', 'n_entités'],
    ascending=[False, False, False, True]
).reset_index(drop=True)
df_metrics.index += 1
df_metrics.index.name = 'ordre de groupe'

print("Métriques de tri internes à _determine_imputation_order :")
display(df_metrics)

In [ ]:
# Visualisation comparative des métriques de tri
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df_metrics))
width = 0.25

b1 = ax.bar(x - width, df_metrics['freq_médiane (ord)'], width, label='Fréquence médiane', color='#4e79a7')
b2 = ax.bar(x,          df_metrics['freq_min (ord)'],    width, label='Fréquence minimale (pire entité)', color='#e15759')
b3 = ax.bar(x + width,  df_metrics['freq_moyenne (ord)'],width, label='Fréquence moyenne', color='#59a14f')

ax.set_xticks(x)
ax.set_xticklabels(df_metrics['variable'], rotation=30, ha='right')
ax.set_ylabel('Ordre de fréquence (plus élevé = fréquence plus basse)')
ax.set_title('Métriques de tri par variable — _determine_imputation_order', fontweight='bold')
ax.legend(fontsize=9)

# Annotation du nombre d'entités
for i, row in df_metrics.iterrows():
    ax.text(x[i-1] + width, row['freq_moyenne (ord)'] + 0.05,
            f"n={row['n_entités']}", ha='center', fontsize=7, color='#666')

plt.tight_layout()
plt.show()

### 💡 Observations

- **`pop_active`** est traitée en premier dans toutes les entités : annuelle partout, sa fréquence médiane est la plus élevée (ordre le plus grand).
- **`pib`** est trimestriel pour `pays_sud` et `pays_est`, mais **annuel** pour `pays_nord`. Sa fréquence médiane se situe entre A et Q — il passe donc après `pop_active` mais *avant* les variables purement trimestrielles dont la médiane est plus basse.
- **`invest`** est mensuel pour `pays_est` — sa fréquence médiane chute vers M, ce qui l'envoie en **fin de liste** parmi les variables à imputer.
- Les variables **propres à une seule entité** (`consomm`, `tx_chomage`, `ipc`) ont `n_entités=1` : elles sont traitées en dernier par rapport aux variables à médiane équivalente, le critère secondaire `n_entities` étant croissant.

---

## 4. Récapitulatif comparatif des trois scénarios

In [ ]:
def format_order_summary(imputer, name):
    """Return a formatted summary string for an imputer's imputation order."""
    lines = [f"── {name} (cible M) ──"]
    for rank, key in enumerate(imputer.imputation_order_, start=1):
        freq = normalize_freq_label(imputer.detected_frequencies_.get(key, '?'))
        label = str(key) if isinstance(key, tuple) else key
        lines.append(f"  #{rank:02d}  [{freq}]  {label}")
    return "\n".join(lines)

print(format_order_summary(imputer_ts,   "Scénario 1 — Séries simples"))
print()
print(format_order_summary(imputer_homo, "Scénario 2 — Panel homogène"))
print()
print(format_order_summary(imputer_het,  "Scénario 3 — Panel hétérogène"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
scenarios = [
    (imputer_ts,   'Scénario 1\nSéries simples'),
    (imputer_homo, 'Scénario 2\nPanel homogène'),
    (imputer_het,  'Scénario 3\nPanel hétérogène'),
]

for ax, (imputer, title) in zip(axes, scenarios):
    order = imputer.imputation_order_
    colors = [FREQ_COLORS.get(normalize_freq_label(imputer.detected_frequencies_.get(k, 'M')), '#aaa')
              for k in order]

    if imputer.is_panel_:
        labels = [f"{k[-1]}\n({k[0]})" if isinstance(k, tuple) else str(k) for k in order]
    else:
        labels = [str(k) for k in order]

    bars = ax.barh(range(len(order)), [len(order) - i for i in range(len(order))],
                   color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Priorité →')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlim(0, len(order) + 0.5)
    for i, bar in enumerate(bars):
        ax.text(bar.get_width() - 0.1, bar.get_y() + bar.get_height() / 2,
                f'#{i+1}', va='center', ha='right', fontsize=8, color='white', fontweight='bold')

patches = [mpatches.Patch(color=FREQ_COLORS['A'], label='Annuelle'),
           mpatches.Patch(color=FREQ_COLORS['Q'], label='Trimestrielle'),
           mpatches.Patch(color=FREQ_COLORS['M'], label='Mensuelle')]
fig.legend(handles=patches, fontsize=9, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Comparaison des ordres d\'imputation — 3 scénarios', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()